# Day 20 — Graph RAG / Knowledge Graph

**Goal:** understand how entity–relation structure supports retrieval and reasoning, and compare Graph RAG with vector RAG.

Core pipeline:

$$
\text{Question}
\rightarrow
\text{Entity Linking}
\rightarrow
\text{Graph Retrieval}
\rightarrow
\text{Subgraph}
\rightarrow
\text{Serialization}
\rightarrow
\text{LLM Reasoning}
\rightarrow
\text{Answer}
$$

Today is still a foundation/research-map day, not a production GraphRAG implementation day.


## 0. What you already know

From Day 19:
- Vector RAG retrieves semantically similar chunks.
- Graph RAG can retrieve connected entities, relations, paths, and subgraphs.
- Graph retrieval and graph reasoning are different.
- A Graph Agent can query a graph through tools such as `graph_lookup(entity, relation)`.


# 1. Knowledge Graph basics

A **Knowledge Graph (KG)** stores knowledge as entities and typed relations.

A common representation is a triple:

$$
(\text{head}, \text{relation}, \text{tail})
$$

Example:

```text
(ReAct, published_at, ICLR 2023)
(ICLR 2023, held_in, Kigali)
(Kigali, located_in, Rwanda)
```

Graphically:

```text
ReAct
  ↓ published_at
ICLR 2023
  ↓ held_in
Kigali
  ↓ located_in
Rwanda
```


## Mandatory Question 1

Explain:
1. entity
2. relation
3. triple
4. why a KG can help with multi-hop questions

Answer:

1. An entity represents an object or concept in a knowledge graph and is usually represented as a node.

2. A relation describes how two entities are connected and usually corresponds to a typed edge in the graph.

3. A triple consists of a head entity, a relation, and a tail entity, such as (ReAct, published_at, ICLR 2023).

4. A KG helps with multi-hop questions because explicit relations can be followed across multiple entities, allowing several connected facts to be combined to derive an answer. 


# 2. Vector RAG vs Graph RAG

## Vector RAG

```text
Question → query embedding → similarity search → top-k chunks → LLM → Answer
```

Main signal:

$$
\text{semantic similarity}(q, c_i)
$$

## Graph RAG

```text
Question
   ↓
entity / relation identification
   ↓
graph traversal / subgraph retrieval
   ↓
structured evidence
   ↓
LLM
   ↓
Answer
```

Graph retrieval can use connectivity, relation types, paths, neighborhoods, and structural constraints.

These approaches are complementary; hybrid systems can use both.


## Mandatory Question 2

Give:
- one task where vector retrieval is probably sufficient,
- one task where graph retrieval is likely more useful.

Explain using **semantic similarity vs relational structure**.

Answer: Vector retrieval is sufficient for a question such as “What is ReAct?”, because the answer can be found from semantically similar text chunks. Graph retrieval is more useful for a multi-hop question such as “In which country was the conference that published ReAct held?”, because the answer depends on following explicit relations between several entities.

# 3. Entity linking

Before retrieving a graph, the system often maps text mentions to graph entities.

Example:

```text
"ReAct" → KG entity: ReAct
```

This is **entity linking**.

```text
Text Mention
   ↓
Candidate Entities
   ↓
Disambiguation
   ↓
Linked KG Entity
```

Wrong entity linking can send the entire downstream pipeline to the wrong part of the graph.


## Mandatory Question 3

Why is entity linking different from graph retrieval?

What happens if `"Apple"` is linked to the fruit entity when the question is actually about Apple Inc.?

Answer: Entity linking identifies which graph entity a text mention refers to, while graph retrieval searches for relevant nodes, relations, paths, or subgraphs starting from the linked entity. If “Apple” is incorrectly linked to the fruit instead of Apple Inc., graph retrieval will search the wrong part of the graph and may return irrelevant evidence, causing downstream reasoning to fail.

# 4. Graph retrieval

After identifying a starting entity, the system retrieves useful graph evidence.

Possible strategies:
- 1-hop neighborhood
- k-hop neighborhood
- relation-constrained traversal
- path search
- graph ranking
- subgraph expansion
- semantic + structural hybrid retrieval

Goal:

> retrieve a subgraph that is small enough for the downstream model but complete enough to contain the required evidence.

Tradeoff:

$$
\text{Evidence Recall}
\leftrightarrow
\text{Context Size / Noise}
$$


## Mandatory Question 4

Why is retrieving the entire graph usually a bad idea?

Explain the tradeoff between retrieving too little and retrieving too much.

Answer: Retrieving the entire graph requires unnecessary computation and memory, increases latency, and introduces a large amount of irrelevant information. Retrieving too little may miss important evidence, while retrieving too much introduces noise and increases computational and context costs. Therefore, the goal is to retrieve a small but sufficiently informative subgraph.

# 5. Subgraph retrieval

A **subgraph** is a selected subset of nodes and edges.

For:

> In which country was the conference that published ReAct held?

a useful subgraph is:

```text
ReAct → ICLR 2023 → Kigali → Rwanda
```

A useful retriever should maximize evidence coverage while minimizing irrelevant structure.


## Mandatory Question 5

Retriever A returns only:

```text
ReAct → ICLR 2023
```

Retriever B returns the correct 3-hop path plus 500 unrelated nodes.

Why can both be problematic? What should a better retriever optimize?

Answer: Retriever A may miss important multi-hop evidence because the retrieved subgraph is incomplete. Retriever B contains the correct evidence but also introduces excessive irrelevant nodes, which increase noise and context cost. A better retriever should retrieve a compact subgraph that preserves the necessary evidence while minimizing irrelevant structure.

# 6. Graph serialization

LLMs consume token sequences, not native graph objects.

A subgraph can be serialized as:

```text
Triples:
(ReAct, published_at, ICLR 2023)

Natural language:
"ReAct was published at ICLR 2023."

Path:
ReAct --published_at--> ICLR 2023

JSON:
{"entity":"ReAct","relation":"published_at","target":"ICLR 2023"}
```

Different serializations can make the same graph easier or harder for an LLM to reason over.


## Mandatory Question 6

Why can two serializations of the same subgraph produce different LLM reasoning performance?

Give one example.

Answer: Different serializations organize the same graph information in different ways. Some formats make relation direction, path structure, and entity connections more explicit, while others may introduce ambiguity or extra tokens. For example, the path ReAct --published_at--> ICLR 2023 --held_in--> Kigali may make the multi-hop relation easier to follow than two loosely written natural-language sentences.

# 7. Graph reasoning after retrieval

Even with perfect retrieval, reasoning can fail.

Possible failures:
- wrong relation direction,
- stopping too early,
- combining unrelated paths,
- ignoring constraints,
- hallucinating a missing edge.

Therefore:

$$
\boxed{\text{Better Retrieval} \neq \text{Guaranteed Better Reasoning}}
$$


## Mandatory Question 7

Construct a case where:
- entity linking succeeds,
- graph retrieval succeeds,
- graph reasoning fails.

Identify the exact failure stage.

Answer: Entity linking succeeds by correctly mapping “ReAct” to the ReAct entity. Graph retrieval also succeeds by retrieving the path ReAct → ICLR 2023 → Kigali → Rwanda. However, the LLM incorrectly answers “Kigali” instead of “Rwanda”. Therefore, the failure occurs at the graph reasoning stage, because the model fails to correctly interpret and combine the retrieved relations.

# 8. Hybrid RAG

```text
                    Question
                       │
          ┌────────────┴────────────┐
          ↓                         ↓
    Vector Retrieval          Graph Retrieval
          ↓                         ↓
     Similar Chunks          Relevant Subgraph
          └────────────┬────────────┘
                       ↓
                 Evidence Fusion
                       ↓
                      LLM
                       ↓
                     Answer
```

Vector retrieval helps with semantic matching and unstructured text.
Graph retrieval helps with explicit relations and multi-hop structure.


## Mandatory Question 8

Why is hybrid RAG not simply “more retrieval is always better”?

What problems can appear when combining vector and graph evidence?

Answer: Hybrid RAG is not simply “more retrieval is better” because additional evidence can also introduce noise, redundancy, conflicting information, and longer context. Vector and graph retrieval use different relevance signals, so the system must decide how to rank, filter, and fuse them. A good hybrid retriever should maximize complementary useful evidence while minimizing irrelevant or conflicting information and context cost.

# 9. Microsoft GraphRAG — what to understand today

High-level idea:

```text
Documents
   ↓
extract entities / relationships / claims
   ↓
build a graph
   ↓
organize / summarize graph structure
   ↓
retrieve graph-derived evidence
   ↓
LLM answers questions
```

Important lesson:

> The graph may be pre-existing, or it may be constructed from unstructured documents.


## Mandatory Question 9

Compare:

### A
A curated Knowledge Graph already exists.

### B
Only raw documents exist, so the system must construct a graph first.

What new failure modes appear in B?

Answer: In B, new failure modes appear during graph construction. The system may extract wrong or missing entities, extract incorrect relations, fail to resolve entities correctly, or create missing/spurious edges. These construction errors can then propagate into graph retrieval and reasoning, even if the retrieval and reasoning components themselves work correctly.

# 10. Research bottlenecks

Potential bottlenecks:

```text
Documents
   ↓
Entity/Relation Extraction
   ↓
Entity Linking
   ↓
Graph Construction
   ↓
Graph Retrieval
   ↓
Subgraph Selection
   ↓
Serialization
   ↓
LLM Reasoning
   ↓
Answer
```

Examples:
- noisy extraction,
- ambiguous entity linking,
- incomplete graph,
- poor multi-hop retrieval,
- oversized subgraph,
- serialization loss,
- weak reasoning,
- error propagation,
- latency/cost.


## Mandatory Question 10 — Research decomposition

Choose one bottleneck and write:

1. Bottleneck
2. Why current systems may fail
3. Intervention
4. Controlled experiment / ablation
5. Metrics

Answer: 
1. Bottleneck:
The retriever may fail to retrieve all necessary nodes and relations for a multi-hop question.

2. Why current systems may fail:
A one-hop or similarity-based retriever may retrieve only locally relevant evidence and miss distant but necessary relations.

3. Intervention:
Use relation-aware multi-hop graph traversal or a learned subgraph retriever to retrieve a more complete but still compact subgraph.

4. Controlled experiment / ablation:
Keep the dataset, questions, LLM, prompts, and reasoning settings fixed. Compare the baseline retriever with the proposed multi-hop retriever.

5. Metrics:
Measure retrieval recall / evidence coverage, subgraph size, final answer accuracy, and optionally latency or token cost.


# 11. Tiny local experiment

This is only a toy demonstration of two different retrieval signals.


In [1]:
chunks = {
    "c1": "ReAct was published at ICLR 2023.",
    "c2": "ICLR 2023 was held in Kigali.",
    "c3": "Kigali is located in Rwanda.",
    "c4": "ReAct combines reasoning and acting in language models.",
    "c5": "Rwanda is a country in East Africa.",
}

graph = {
    ("ReAct", "published_at"): "ICLR 2023",
    ("ICLR 2023", "held_in"): "Kigali",
    ("Kigali", "located_in"): "Rwanda",
}

question = "In which country was the conference that published ReAct held?"
print(question)


In which country was the conference that published ReAct held?


### Weak lexical retriever

This uses word overlap as a toy stand-in for semantic similarity.


In [2]:
import re

def tokenize(text):
    return set(re.findall(r"[A-Za-z0-9]+", text.lower()))

def lexical_score(query, text):
    return len(tokenize(query) & tokenize(text))

ranked = sorted(
    chunks.items(),
    key=lambda kv: lexical_score(question, kv[1]),
    reverse=True
)

for cid, text in ranked:
    print(cid, lexical_score(question, text), "->", text)


c1 3 -> ReAct was published at ICLR 2023.
c2 3 -> ICLR 2023 was held in Kigali.
c4 2 -> ReAct combines reasoning and acting in language models.
c5 2 -> Rwanda is a country in East Africa.
c3 1 -> Kigali is located in Rwanda.


### Graph traversal


In [3]:
def graph_lookup(entity, relation):
    return graph.get((entity, relation), "NOT_FOUND")

venue = graph_lookup("ReAct", "published_at")
city = graph_lookup(venue, "held_in")
country = graph_lookup(city, "located_in")

print("venue:", venue)
print("city:", city)
print("country:", country)


venue: ICLR 2023
city: Kigali
country: Rwanda


Toy lesson:

```text
lexical/vector-style retrieval → ranks by similarity
graph retrieval → follows explicit relational structure
```

This does **not** prove that Graph RAG is always better.


## Mandatory Question 11

What does the toy experiment demonstrate, and what does it not prove?

Answer: The toy experiment demonstrates that lexical/vector-style retrieval and graph retrieval use different retrieval signals. Lexical retrieval ranks chunks based on similarity to the query, while graph retrieval follows explicit relational structure and can recover multi-hop evidence that may not be highly similar to the original question. However, the experiment does not prove that Graph RAG is always better than Vector RAG, because the lexical retriever is extremely simplified and real vector retrieval systems use much stronger semantic embeddings and reranking methods.

# 12. Day 20 Final Recap

1. What is a Knowledge Graph?
2. What is a triple?
3. What is entity linking?
4. What is graph retrieval?
5. What is a subgraph?
6. Why not retrieve the entire graph?
7. What is graph serialization?
8. Why can retrieval succeed while reasoning fails?
9. Difference between vector RAG and Graph RAG?
10. Why can hybrid RAG help?
11. Where can error propagation occur?
12. Name one Graph RAG bottleneck that interests you.

**One-sentence target**

> Graph RAG augments LLM generation with graph-structured evidence by linking queries to entities, retrieving relevant relational subgraphs, and exposing that structured evidence to downstream reasoning.


# Related Learning Materials

## Required today

### Microsoft GraphRAG
- Documentation: https://microsoft.github.io/graphrag/
- GitHub: https://github.com/microsoft/graphrag

Read only:
- overview / introduction
- high-level architecture
- entity/relationship extraction
- local/global retrieval idea if clearly presented

## Knowledge Graph refresher
- Stanford CS224W: https://web.stanford.edu/class/cs224w/

Use only if you want to refresh entities, relation types, triples, and relational graph ideas.

## Optional surveys
- KG + LLM survey search: https://arxiv.org/search/?query=knowledge+graph+large+language+model+survey&searchtype=all
- Graph RAG survey search: https://arxiv.org/search/?query=graph+rag+survey&searchtype=all

Do not deep-read multiple surveys today.

## Suggested schedule

- KG + Graph RAG concepts: **60–90 min**
- Microsoft GraphRAG overview: **30–45 min**
- Toy experiment + questions: **30–45 min**
- Final recap: **20–30 min**

Day 20 is complete when you can explain:

```text
entity linking
→ graph retrieval
→ subgraph
→ serialization
→ LLM reasoning
```

and compare it clearly with vector RAG.
